In [1]:
import pandas as pd

# Load the raw IoT sensor data into a pandas DataFrame
df = pd.read_csv('/content/iot_sensor_raw_data_extended.csv')

# Display the first 5 rows of the DataFrame
display(df.head())

,timestamp,device_id,temp_c,motion,battery
0,2026-07-12 14:50:00,SN-9982,28.3,false,88.0
1,2026-07-12 14:50:00,SN-9983,28.2,true,94.0
2,2026-07-12 14:50:00,SN-9984,29.7,true,76.0
3,2026-07-12 14:50:00,SN-9985,27.8,true,81.0
4,2026-07-12 14:50:00,SN-9986,30.7,true,69.0


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   timestamp  480 non-null    object 
 1   device_id  478 non-null    object 
 2   temp_c     473 non-null    float64
 3   motion     476 non-null    object 
 4   battery    476 non-null    float64
dtypes: float64(2), object(3)
memory usage: 18.9+ KB


In [3]:
print('Missing values per column:')
display(df.isnull().sum())

Missing values per column:


,0
timestamp,0
device_id,2
temp_c,7
motion,4
battery,4


### Data Quality Rules:
1.  **Temperature (`temp_c`)**: Must be between 0–50 degrees Celsius.
2.  **Device ID (`device_id`)**: Cannot be null/empty.
3.  **Duplicates**: No duplicate rows based on the combination of `device_id` and `timestamp`.
4.  **Battery (`battery`)**: Must be between 0–100.

In [4]:
print('Checking for duplicate rows based on device_id and timestamp:')
duplicate_rows = df[df.duplicated(subset=['device_id', 'timestamp'], keep=False)]
print(f'Number of duplicate rows: {len(duplicate_rows)}')
display(duplicate_rows.sort_values(by=['device_id', 'timestamp']))

Checking for duplicate rows based on device_id and timestamp:
Number of duplicate rows: 50


,timestamp,device_id,temp_c,motion,battery
92,2026-07-12 14:53:00,SN-9982,29.4,true,88.0
93,2026-07-12 14:53:00,SN-9982,29.4,true,88.0
113,2026-07-12 14:53:40,SN-9982,29.4,false,88.0
114,2026-07-12 14:53:40,SN-9982,29.4,false,88.0
181,2026-07-12 14:55:50,SN-9982,NaN,true,87.0
182,2026-07-12 14:55:50,SN-9982,NaN,true,87.0
222,2026-07-12 14:57:10,SN-9982,28.1,false,87.0
223,2026-07-12 14:57:10,SN-9982,28.1,false,87.0
282,2026-07-12 14:59:00,SN-9982,27.4,true,86.0
283,2026-07-12 14:59:00,SN-9982,27.4,true,86.0


In [5]:
initial_rows = len(df)
df.drop_duplicates(subset=['device_id', 'timestamp'], inplace=True)
rows_after_removing_duplicates = len(df)

print(f'Number of rows before removing duplicates: {initial_rows}')
print(f'Number of rows after removing duplicates: {rows_after_removing_duplicates}')
print(f'Number of duplicate rows removed: {initial_rows - rows_after_removing_duplicates}')

Number of rows before removing duplicates: 480
Number of rows after removing duplicates: 455
Number of duplicate rows removed: 25


In [6]:
print('Rows with missing device_id before cleaning:')
display(df[df['device_id'].isnull()])

initial_rows_after_duplicates = len(df)
df.dropna(subset=['device_id'], inplace=True)
rows_after_device_id_cleanup = len(df)

print(f'Number of rows before removing missing device_id: {initial_rows_after_duplicates}')
print(f'Number of rows after removing missing device_id: {rows_after_device_id_cleanup}')
print(f'Number of rows removed due to missing device_id: {initial_rows_after_duplicates - rows_after_device_id_cleanup}')

Rows with missing device_id before cleaning:


,timestamp,device_id,temp_c,motion,battery
305,2026-07-12 14:59:40,NaN,26.4,true,79.0
385,2026-07-12 15:02:10,NaN,28.4,true,85.0


Number of rows before removing missing device_id: 455
Number of rows after removing missing device_id: 453
Number of rows removed due to missing device_id: 2


In [7]:
print('Rows with temp_c outside 0-50 degrees Celsius:')
outlier_temp_rows = df[(df['temp_c'] < 0) | (df['temp_c'] > 50)]
print(f'Number of temperature outliers: {len(outlier_temp_rows)}')
display(outlier_temp_rows)

Rows with temp_c outside 0-50 degrees Celsius:
Number of temperature outliers: 10


,timestamp,device_id,temp_c,motion,battery
83,2026-07-12 14:52:40,SN-9983,-999.0,true,94.0
127,2026-07-12 14:54:00,SN-9984,-999.0,true,75.0
178,2026-07-12 14:55:40,SN-9984,85.0,true,75.0
194,2026-07-12 14:56:10,SN-9984,-999.0,true,75.0
253,2026-07-12 14:58:00,SN-9985,85.0,true,79.0
267,2026-07-12 14:58:30,SN-9983,85.0,false,92.0
404,2026-07-12 15:02:40,SN-9984,-999.0,true,73.0
414,2026-07-12 15:03:00,SN-9983,-999.0,true,91.0
419,2026-07-12 15:03:10,SN-9983,67.8,true,91.0
446,2026-07-12 15:04:00,SN-9983,-999.0,true,91.0


In [8]:
df.loc[(df['temp_c'] < 0) | (df['temp_c'] > 50), 'temp_c'] = None
print('Temperature outliers have been converted to NULL.')

# Verify the change by checking for outliers again
print('Rows with temp_c outside 0-50 degrees Celsius after conversion:')
outlier_temp_rows_after_conversion = df[(df['temp_c'] < 0) | (df['temp_c'] > 50)]
print(f'Number of temperature outliers after conversion: {len(outlier_temp_rows_after_conversion)}')
display(outlier_temp_rows_after_conversion)

Temperature outliers have been converted to NULL.
Rows with temp_c outside 0-50 degrees Celsius after conversion:
Number of temperature outliers after conversion: 0


,timestamp,device_id,temp_c,motion,battery


In [9]:
print('Rows with battery outside 0-100:')
outlier_battery_rows = df[(df['battery'] < 0) | (df['battery'] > 100)]
print(f'Number of battery outliers: {len(outlier_battery_rows)}')
display(outlier_battery_rows)

Rows with battery outside 0-100:
Number of battery outliers: 5


,timestamp,device_id,temp_c,motion,battery
42,2026-07-12 14:51:20,SN-9983,28.3,true,-5.0
180,2026-07-12 14:55:40,SN-9986,29.5,true,-1.0
235,2026-07-12 14:57:30,SN-9983,27.2,false,999.0
252,2026-07-12 14:58:00,SN-9984,28.1,false,115.0
426,2026-07-12 15:03:20,SN-9984,30.0,false,108.0


In [10]:
df.loc[(df['battery'] < 0) | (df['battery'] > 100), 'battery'] = None
print('Battery outliers have been converted to NULL.')

# Verify the change by checking for outliers again
print('Rows with battery outside 0-100 after conversion:')
outlier_battery_rows_after_conversion = df[(df['battery'] < 0) | (df['battery'] > 100)]
print(f'Number of battery outliers after conversion: {len(outlier_battery_rows_after_conversion)}')
display(outlier_battery_rows_after_conversion)

Battery outliers have been converted to NULL.
Rows with battery outside 0-100 after conversion:
Number of battery outliers after conversion: 0


,timestamp,device_id,temp_c,motion,battery


In [11]:
print('Missing values per column after outlier conversion:')
display(df.isnull().sum())

Missing values per column after outlier conversion:


,0
timestamp,0
device_id,0
temp_c,15
motion,4
battery,8


In [12]:
# Impute missing 'temp_c' with its mean
df['temp_c'] = df['temp_c'].fillna(df['temp_c'].mean())

# Impute missing 'battery' with its mean
df['battery'] = df['battery'].fillna(df['battery'].mean())

# Impute missing 'motion' with its mode
# Note: mode() can return multiple values if there's a tie, so we take the first one.
df['motion'] = df['motion'].fillna(df['motion'].mode()[0])

print('Missing values have been imputed for temp_c, motion, and battery.')

# Verify no more missing values
print('\nMissing values per column after imputation:')
display(df.isnull().sum())

Missing values have been imputed for temp_c, motion, and battery.

Missing values per column after imputation:


,0
timestamp,0
device_id,0
temp_c,0
motion,0
battery,0


In [13]:
# Add data_quality_status column
# Since all cleaning steps have been performed, and the DataFrame now adheres to all defined rules,
# we can mark all remaining rows as 'Clean'.
df['data_quality_status'] = 'Clean'

print('Added data_quality_status column with status "Clean" for all rows.')
display(df.head())

Added data_quality_status column with status "Clean" for all rows.


,timestamp,device_id,temp_c,motion,battery,data_quality_status
0,2026-07-12 14:50:00,SN-9982,28.3,false,88.0,Clean
1,2026-07-12 14:50:00,SN-9983,28.2,true,94.0,Clean
2,2026-07-12 14:50:00,SN-9984,29.7,true,76.0,Clean
3,2026-07-12 14:50:00,SN-9985,27.8,true,81.0,Clean
4,2026-07-12 14:50:00,SN-9986,30.7,true,69.0,Clean


In [14]:
final_rows = len(df)

print(f'Initial number of rows (raw data): {initial_rows}')
print(f'Final number of rows (cleaned data): {final_rows}')
print(f'Total rows removed during cleaning: {initial_rows - final_rows}')

Initial number of rows (raw data): 480
Final number of rows (cleaned data): 453
Total rows removed during cleaning: 27


In [15]:
# Save the cleaned DataFrame to a new CSV file
output_filename = 'iot_sensor_cleaned_data.csv'
df.to_csv(output_filename, index=False)

print(f'Cleaned data saved to {output_filename}')

Cleaned data saved to iot_sensor_cleaned_data.csv
